In [ ]:
#!/usr/bin/env python3
"""
AMOC RECONSTRUCTION PIPELINE (Marion split)
===========================================

Goal:
  - Discover smallest interpretable set of latent predictors (EOF PCs)
  - Train and select features ONLY in 1850–1950 (TRAIN)
  - Evaluate on 1951–1980 (TEST)
  - Apply to 1981–2014 (LATE diagnostic: relationship breakdown)

Key design:
  - CV for hyperparameters happens ONLY inside TRAIN
  - Feature ranking happens ONLY inside fold-train (within TRAIN CV)
  - Final ranking/features derived ONLY from full TRAIN
  - TEST and LATE are never used for ranking or hyperparameter selection

Outputs saved with save_tag:
  - *_final_features.csv
  - *_train_feature_ranking.csv
  - *_cv_grid_results.csv
  - *_best_params.csv
  - *_test_predictions.csv
  - *_late_predictions.csv
  - *_test_metrics.csv
  - *_late_metrics.csv
"""

import os
import numpy as np
import pandas as pd
import xarray as xr

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score


# ============================================================
# USER CONFIG
# ============================================================

MODEL  = "IPSL-CM6A-LR"
TARGET = "AMOC_45N_ensmean"     # e.g. "AMOC_45N_ensmean" or "AMOC_45N_member"
MODE   = "ensmean"              # "ensmean" or "member"
MEMBERS = [0, 1, 2, 3, 4]        # used only if MODE="member"

CFG = dict(
    # paths
    EOF_DIR_TEMPLATE="/data/projects/nckf/frekle/EOF_results/{MODEL}/latdepth_sections/",
    AMOC_FILE_TEMPLATE="/data/users/frekle/AMOC_analysis/AMOC_{MODEL}.nc",
    OUT_ROOT="/data/users/frekle/AMOC_analysis/PIPELINE_RESULTS_NOTEBOOK/",

    # common year filtering
    YEAR_START=1850,
    YEAR_END=2014,

    # ✅ Marion split
    TRAIN_START=1850, TRAIN_END=1950,
    TEST_START=1951,  TEST_END=1980,
    LATE_START=1981,  LATE_END=2014,

    # CV (inside TRAIN only)
    N_SPLITS=5,
    GAP_EMBARGO=10,         # should be <= MAX_LAG_ALLOWED
    MIN_CORR_POINTS=30,

    # predictor space
    VARS=["thetao", "so"],
    LON_TAGS=["W10p0", "W20p0", "W30p0", "W40p0", "W60p0"],
    N_MODES=10,

    # lags
    MAX_LAG_TOTAL=50,       # total lags considered in ranking
    MAX_LAG_ALLOWED=20,     # restrict to <= this in ranking + feature building
    CAP_LAGS_PER_GROUP=None,

    # hyperparameter grids
    N_LIST=[5, 10, 15],
    W_LIST=[0, 1],
    RIDGE_ALPHAS=[1, 10, 100, 1000, 1e4],

    USE_ONE_SE_RULE=True,
)

OUTDIR = os.path.join(CFG["OUT_ROOT"], MODEL, TARGET, f"mode_{MODE}")
os.makedirs(OUTDIR, exist_ok=True)
print("OUTDIR:", OUTDIR)


# ============================================================
# HELPERS
# ============================================================

def extract_years(time_coord):
    try:
        return xr.DataArray(time_coord).dt.year.values.astype(int)
    except Exception:
        t = np.asarray(time_coord)
        return np.array([int(str(x)[:4]) for x in t], dtype=int)

def years_to_idx(years, y0, y1):
    years = np.asarray(years, dtype=int)
    return np.where((years >= int(y0)) & (years <= int(y1)))[0]

def load_pc(EOF_DIR, var, lon_tag, n_modes, member_mode=False, member_id=None):
    f = os.path.join(EOF_DIR, f"EOF_latdepth_{var}_{lon_tag}.nc")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")

    ds = xr.open_dataset(f)
    PC = ds["PC"].isel(mode=slice(0, n_modes))

    if "member" in PC.dims:
        if member_mode:
            if member_id is None:
                raise ValueError("member_id must be provided for member_mode=True")
            # allow label or integer
            if "member" in PC.coords and member_id in PC["member"].values:
                PC = PC.sel(member=member_id)
            else:
                PC = PC.isel(member=int(member_id))
        else:
            PC = PC.mean("member")

    PC = PC.transpose("time", "mode")
    years = extract_years(PC["time"])
    PC = PC.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")
    return PC.astype(float)  # (year, mode)

def load_amoc(AMOC_FILE, target, member_mode=False, member_id=None):
    dsA = xr.open_dataset(AMOC_FILE)
    if target not in dsA.variables:
        raise KeyError(f"Target '{target}' not found. Available: {list(dsA.data_vars)}")

    am = dsA[target].squeeze()

    if "member" in am.dims:
        if member_mode:
            if member_id is None:
                raise ValueError("member_id must be provided for member_mode=True")
            if "member" in am.coords and member_id in am["member"].values:
                am = am.sel(member=member_id)
            else:
                am = am.isel(member=int(member_id))
        else:
            am = am.mean("member")

    # convert to year axis
    if "year" in am.dims:
        am_year = am
        if "time" in am_year.coords:
            am_year = am_year.drop_vars("time")
    else:
        years = extract_years(am["time"])
        am_year = am.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")

    am_year = am_year.astype(float).squeeze()
    if am_year.ndim != 1:
        raise ValueError(f"AMOC target not 1D after selection: shape={am_year.shape}")
    return am_year  # (year,)

def corr_1d(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 3:
        return np.nan
    aa = a[m] - np.mean(a[m])
    bb = b[m] - np.mean(b[m])
    denom = np.sqrt(np.sum(aa**2) * np.sum(bb**2))
    if denom == 0:
        return np.nan
    return float(np.sum(aa * bb) / denom)

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true).squeeze()
    y_pred = np.asarray(y_pred).squeeze()
    r2 = float(r2_score(y_true, y_pred)) if len(y_true) >= 2 else np.nan
    r  = float(np.corrcoef(y_true, y_pred)[0, 1]) if (np.std(y_true) > 0 and np.std(y_pred) > 0) else np.nan
    var_ratio = float(np.std(y_pred) / np.std(y_true)) if np.std(y_true) > 0 else np.nan
    return r2, r, var_ratio

def rank_features_train_only(pc_np_sub, y_sub, vars_, lons_, n_modes, lags_rank):
    rows = []
    Tsub = len(y_sub)
    for var in vars_:
        for lon in lons_:
            pc = pc_np_sub[(var, lon)]
            for lag in lags_rank:
                lag = int(lag)
                if lag >= Tsub:
                    continue
                t_idx = np.arange(lag, Tsub)
                p_idx = t_idx - lag
                yy = y_sub[t_idx]
                for m in range(n_modes):
                    pp = pc[p_idx, m]
                    c = corr_1d(pp, yy)
                    if np.isfinite(c):
                        rows.append({
                            "var": var, "lon": lon, "mode": int(m+1),
                            "lag": int(lag), "corr": float(c), "abs_corr": float(abs(c))
                        })
    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise RuntimeError("No finite correlations computed.")
    return df.sort_values("abs_corr", ascending=False).reset_index(drop=True)

def build_feature_set(df_ranked, n_seeds, W, max_lag_allowed, cap_lags_per_group=None):
    df_top = df_ranked.head(int(n_seeds))
    seeds = [(str(r.var), str(r.lon), int(r.mode)-1, int(r.lag)) for r in df_top.itertuples(index=False)]

    expanded = []
    for (var, lon, m0, lag0) in seeds:
        for L in range(lag0 - int(W), lag0 + int(W) + 1):
            if L < 0:
                continue
            if max_lag_allowed is not None and L > int(max_lag_allowed):
                continue
            expanded.append((var, lon, m0, int(L)))

    if cap_lags_per_group is not None:
        keep = []
        seen = {}
        for feat in expanded:
            key = (feat[0], feat[1], feat[2])
            seen.setdefault(key, set())
            if feat[3] not in seen[key]:
                if len(seen[key]) < int(cap_lags_per_group):
                    seen[key].add(feat[3])
                    keep.append(feat)
        expanded = keep

    return sorted(set(expanded), key=lambda x: (x[0], x[1], x[2], x[3]))

def build_XY(pc_np_full, y_full, feats, t_positions):
    t_positions = np.asarray(t_positions, dtype=int)
    max_lag = max(lag for (_, _, _, lag) in feats) if len(feats) else 0
    used = t_positions[t_positions >= max_lag]
    if len(used) == 0:
        return np.empty((0, len(feats))), np.empty((0,)), used, int(max_lag)

    X = np.empty((len(used), len(feats)), dtype=float)
    for i, t in enumerate(used):
        for j, (var, lon, m0, lag) in enumerate(feats):
            X[i, j] = pc_np_full[(var, lon)][t - lag, m0]
    Y = y_full[used]
    return X, Y, used, int(max_lag)

def make_gap_folds(T_train, n_splits, gap, max_lag_allowed, min_corr_points=30):
    base = np.arange(T_train, dtype=int)
    tscv = TimeSeriesSplit(n_splits=int(n_splits))
    folds = []
    for tr, va in tscv.split(base):
        if gap is not None and gap > 0:
            if len(tr) <= gap:
                continue
            tr = tr[:-int(gap)]

        if max_lag_allowed is not None:
            if len(tr) - int(max_lag_allowed) < int(min_corr_points):
                continue

        if len(tr) < 20 or len(va) < 10:
            continue

        folds.append((tr, va))
    return folds

def one_se_rule_choose(df_grid):
    best = df_grid.iloc[0]
    threshold = best["mean_val_r2"] - best["std_val_r2"]
    ok = df_grid[df_grid["mean_val_r2"] >= threshold].copy()
    if len(ok) == 0:
        return best.to_dict()
    ok = ok.sort_values(["N_seeds","W","alpha"], ascending=[True,True,False]).reset_index(drop=True)
    return ok.iloc[0].to_dict()


# ============================================================
# CORE RUN
# ============================================================

def run_pipeline_one(model, target, mode, cfg, member_id=None, outdir=None, verbose=True):
    EOF_DIR = cfg["EOF_DIR_TEMPLATE"].format(MODEL=model)
    AMOC_FILE = cfg["AMOC_FILE_TEMPLATE"].format(MODEL=model)

    # ----- Load PCs for all requested VARS/LON_TAGS -----
    pc_dict = {}
    for var in cfg["VARS"]:
        for lon in cfg["LON_TAGS"]:
            pc_dict[(var, lon)] = load_pc(
                EOF_DIR, var, lon, cfg["N_MODES"],
                member_mode=(mode=="member"),
                member_id=member_id
            )

    # ----- Load AMOC -----
    am = load_amoc(AMOC_FILE, target, member_mode=(mode=="member"), member_id=member_id)

    # ----- Align years -----
    common_years = am["year"].values.astype(int)
    for da in pc_dict.values():
        common_years = np.intersect1d(common_years, da["year"].values.astype(int))
    common_years = np.asarray(common_years, int)
    common_years.sort()
    common_years = common_years[(common_years >= cfg["YEAR_START"]) & (common_years <= cfg["YEAR_END"])]

    y_full = am.sel(year=common_years).values.astype(float).squeeze()
    pc_np_full = {k: v.sel(year=common_years).values.astype(float) for k, v in pc_dict.items()}

    # ----- Make year-index splits -----
    idx_train = years_to_idx(common_years, cfg["TRAIN_START"], cfg["TRAIN_END"])
    idx_test  = years_to_idx(common_years, cfg["TEST_START"],  cfg["TEST_END"])
    idx_late  = years_to_idx(common_years, cfg["LATE_START"],  cfg["LATE_END"])

    if verbose:
        print(f"\n✓ YEARS total: {common_years[0]}–{common_years[-1]} (T={len(common_years)})")
        print(f"✓ TRAIN: {cfg['TRAIN_START']}–{cfg['TRAIN_END']} (T={len(idx_train)})")
        print(f"✓ TEST:  {cfg['TEST_START']}–{cfg['TEST_END']} (T={len(idx_test)})")
        print(f"✓ LATE:  {cfg['LATE_START']}–{cfg['LATE_END']} (T={len(idx_late)})")

    # lags used in ranking (restricted)
    lags = np.arange(0, cfg["MAX_LAG_TOTAL"], dtype=int)
    if cfg["MAX_LAG_ALLOWED"] is not None:
        lags = lags[lags <= int(cfg["MAX_LAG_ALLOWED"])]

    # CV folds within TRAIN only
    folds = make_gap_folds(
        T_train=len(idx_train),
        n_splits=cfg["N_SPLITS"],
        gap=cfg["GAP_EMBARGO"],
        max_lag_allowed=cfg["MAX_LAG_ALLOWED"],
        min_corr_points=cfg["MIN_CORR_POINTS"]
    )

    if verbose:
        print("FOLDS USED:", len(folds))
        for i, (tr, va) in enumerate(folds):
            print(i, "train:", len(tr), "val:", len(va))
        print("\nRunning TimeSeriesSplit CV on TRAIN only...")

    grid_rows = []
    for N_seeds in cfg["N_LIST"]:
        for W in cfg["W_LIST"]:
            for alpha in cfg["RIDGE_ALPHAS"]:
                r2s, cors, vrs = [], [], []

                for (tr_rel, va_rel) in folds:
                    tr_idx = idx_train[tr_rel]
                    va_idx = idx_train[va_rel]

                    y_tr = y_full[tr_idx]
                    pc_tr = {(var, lon): pc_np_full[(var, lon)][tr_idx, :]
                             for var in cfg["VARS"] for lon in cfg["LON_TAGS"]}

                    df_rank_fold = rank_features_train_only(
                        pc_tr, y_tr, cfg["VARS"], cfg["LON_TAGS"], cfg["N_MODES"], lags
                    )

                    feats = build_feature_set(
                        df_rank_fold,
                        n_seeds=N_seeds,
                        W=W,
                        max_lag_allowed=cfg["MAX_LAG_ALLOWED"],
                        cap_lags_per_group=cfg["CAP_LAGS_PER_GROUP"]
                    )

                    Xtr, Ytr, _, _ = build_XY(pc_np_full, y_full, feats, tr_idx)
                    Xva, Yva, _, _ = build_XY(pc_np_full, y_full, feats, va_idx)

                    if len(Ytr) < 20 or len(Yva) < 10:
                        continue

                    mdl = make_pipeline(StandardScaler(), Ridge(alpha=float(alpha)))
                    mdl.fit(Xtr, Ytr)
                    pred = mdl.predict(Xva)

                    r2, r, vr = metrics(Yva, pred)
                    r2s.append(r2); cors.append(r); vrs.append(vr)

                if len(r2s) < max(2, len(folds)-1):
                    continue

                mean_r2 = float(np.mean(r2s))
                std_r2  = float(np.std(r2s))

                if verbose:
                    print(f"  N={N_seeds:2d} W={W} alpha={alpha:<7g} | val R²={mean_r2: .4f} ±{std_r2:.4f} (folds={len(r2s)})")

                grid_rows.append({
                    "N_seeds": int(N_seeds),
                    "W": int(W),
                    "alpha": float(alpha),
                    "mean_val_r2": mean_r2,
                    "std_val_r2": std_r2,
                    "mean_val_corr": float(np.mean(cors)),
                    "mean_val_var_ratio": float(np.mean(vrs)),
                    "n_folds_used": int(len(r2s)),
                })

    df_grid = pd.DataFrame(grid_rows).sort_values(["mean_val_r2","std_val_r2"], ascending=[False, True]).reset_index(drop=True)
    if len(df_grid) == 0:
        raise RuntimeError("No CV results produced. Reduce gap, reduce lags, or loosen grids.")

    best = one_se_rule_choose(df_grid) if cfg["USE_ONE_SE_RULE"] else df_grid.iloc[0].to_dict()

    # Final ranking on full TRAIN only
    y_tr_full = y_full[idx_train]
    pc_tr_full = {(var, lon): pc_np_full[(var, lon)][idx_train, :]
                  for var in cfg["VARS"] for lon in cfg["LON_TAGS"]}
    df_rank_train = rank_features_train_only(pc_tr_full, y_tr_full, cfg["VARS"], cfg["LON_TAGS"], cfg["N_MODES"], lags)

    feats_final = build_feature_set(
        df_rank_train,
        n_seeds=int(best["N_seeds"]),
        W=int(best["W"]),
        max_lag_allowed=cfg["MAX_LAG_ALLOWED"],
        cap_lags_per_group=cfg["CAP_LAGS_PER_GROUP"]
    )

    # Fit on TRAIN
    Xtr, Ytr, used_tr, maxlag_final = build_XY(pc_np_full, y_full, feats_final, idx_train)
    final_model = make_pipeline(StandardScaler(), Ridge(alpha=float(best["alpha"])))
    final_model.fit(Xtr, Ytr)

    # Evaluate on TEST
    Xte, Yte, used_te, _ = build_XY(pc_np_full, y_full, feats_final, idx_test)
    pred_te = final_model.predict(Xte)
    test_r2, test_corr, test_var = metrics(Yte, pred_te)

    # Diagnostic on LATE
    Xla, Yla, used_la, _ = build_XY(pc_np_full, y_full, feats_final, idx_late)
    pred_la = final_model.predict(Xla)
    late_r2, late_corr, late_var = metrics(Yla, pred_la)

    out = dict(
        df_grid=df_grid,
        best=best,
        df_rank_train=df_rank_train,
        feats_final=feats_final,
        years=common_years,
        idx_train=idx_train,
        idx_test=idx_test,
        idx_late=idx_late,
        test_metrics=dict(test_r2=test_r2, test_corr=test_corr, test_var_ratio=test_var,
                          max_lag_used=maxlag_final, n_features_used=len(feats_final)),
        late_metrics=dict(late_r2=late_r2, late_corr=late_corr, late_var_ratio=late_var,
                          max_lag_used=maxlag_final, n_features_used=len(feats_final)),
        test_pred_df=pd.DataFrame({"year": common_years[used_te], "y_true": Yte, "y_pred": pred_te}),
        late_pred_df=pd.DataFrame({"year": common_years[used_la], "y_true": Yla, "y_pred": pred_la}),
        member_id=member_id,
    )
    return out


def save_run(res, outdir, model, target, mode, member_id=None):
    # use TEST skill in tag (Marion plan)
    save_tag = f"TESTR2_{res['test_metrics']['test_r2']:.3f}_N{int(res['best']['N_seeds'])}_W{int(res['best']['W'])}_A{float(res['best']['alpha']):g}"
    if member_id is not None:
        save_tag += f"_member{member_id}"

    pipe_dir = outdir
    os.makedirs(pipe_dir, exist_ok=True)

    # CV grid + params
    res["df_grid"].to_csv(os.path.join(pipe_dir, f"{save_tag}_cv_grid_results.csv"), index=False)
    pd.DataFrame([res["best"]]).to_csv(os.path.join(pipe_dir, f"{save_tag}_best_params.csv"), index=False)

    # ranking (TRAIN)
    res["df_rank_train"].to_csv(os.path.join(pipe_dir, f"{save_tag}_train_feature_ranking.csv"), index=False)

    # final features
    df_feats = pd.DataFrame([{"var": v, "lon": lon, "mode": m+1, "mode0": m, "lag": lag}
                             for (v, lon, m, lag) in res["feats_final"]])
    df_feats.to_csv(os.path.join(pipe_dir, f"{save_tag}_final_features.csv"), index=False)

    # predictions + metrics
    res["test_pred_df"].to_csv(os.path.join(pipe_dir, f"{save_tag}_test_predictions.csv"), index=False)
    res["late_pred_df"].to_csv(os.path.join(pipe_dir, f"{save_tag}_late_predictions.csv"), index=False)

    pd.DataFrame([res["test_metrics"]]).to_csv(os.path.join(pipe_dir, f"{save_tag}_test_metrics.csv"), index=False)
    pd.DataFrame([res["late_metrics"]]).to_csv(os.path.join(pipe_dir, f"{save_tag}_late_metrics.csv"), index=False)

    print("\n✅ Saved run outputs with tag:", save_tag)
    print("   folder:", pipe_dir)
    return save_tag


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    if MODE == "ensmean":
        res = run_pipeline_one(MODEL, TARGET, "ensmean", CFG, member_id=None, outdir=OUTDIR, verbose=True)
        print("\nBEST:", res["best"])
        print("TEST:", res["test_metrics"])
        print("LATE:", res["late_metrics"])
        _ = save_run(res, OUTDIR, MODEL, TARGET, MODE, member_id=None)

    else:
        for mid in MEMBERS:
            print(f"\n===== MEMBER {mid} =====")
            try:
                res = run_pipeline_one(MODEL, TARGET, "member", CFG, member_id=mid, outdir=OUTDIR, verbose=True)
                print("BEST:", res["best"])
                print("TEST:", res["test_metrics"])
                print("LATE:", res["late_metrics"])
                _ = save_run(res, OUTDIR, MODEL, TARGET, MODE, member_id=mid)
            except Exception as e:
                print(f"⚠️ member {mid} failed: {e}")
